# Rule scores — add per-fraud-type columns to any dataset (view only)

Takes any dataset that has the `cust_*` behavioural features and **auto-adds** the rule-layer columns:
applies `RULES` → per-feature `_signal` → per-fraud-type hit/score columns.

Fraud types: **FPF, TrueName, Synthetic, Mule, ATO, Combined (= all)**, each with `_hits`, `_score`,
`_hits_with_uncertain`, `_score_with_uncertain`.

Nothing is written — the demo cell just `describe()`s the score columns so you can look.
The last section is a one-off cleanup that deletes the mistakenly-saved `*_flagged` folders.

In [1]:
%%pyspark project.spark.compatibility
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("rule_scores_view")
    .config("spark.executor.memory", "18g")
    .config("spark.driver.memory", "10g")
    .config("spark.executor.cores", "4")
    .config("spark.sql.shuffle.partitions", "300")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate()
)

Executing for connection type: SPARK_GLUE, connection name: project.spark.compatibility
Creating Glue session...
Create session for connection: project.spark.compatibility


'Session ae2xwmv6edkt56-edd20756-79be-48d9-8e1e-3fa440a91fa5 has been created.'

<sagemaker_studio_dataengineering_sessions.sagemaker_base_session_manager.common.debugging_utils.SessionInfoTableDisplay object>

Session created for connection: project.spark.compatibility.

Connection: project.spark.compatibility | Run start time: 2026-06-22 15:26:16.399043 | Run duration : 0:02:48.591223s.


In [2]:
%%pyspark project.spark.compatibility
# ─── RULES ─────────────────────────────────────────────────────────
# format: feature → {fraud: (op, val), normal: (op, val)}
# uncertain = NOT fraud AND NOT normal AND value IS NOT NULL

RULES = {
    "cust_account_suspension_days_for_last_5_events": {"fraud": (">=", 184), "normal": ("<", 158.5)},
    "cust_account_suspension_k_event_velocity_5": {"fraud": ("<=", 0.0272), "normal": (">", 0.0323)},
    "cust_mno_port_cumulative_event_slope": {"fraud": ("<=", 0.0036), "normal": (">", 0.0102)},
    "cust_mno_port_days_for_last_3_events": {"fraud": ("<=", 6), "normal": (">", 53)},
    "cust_mno_port_interval_cv": {"fraud": (">=", 1.7321), "normal": ("<", 1.4142)},
    "cust_mno_port_interval_max": {"fraud": ("<=", 4), "normal": (">", 29)},
    "cust_mno_port_interval_mean": {"fraud": ("<=", 4), "normal": (">", 23)},
    "cust_mno_port_interval_median": {"fraud": ("<=", 3), "normal": (">", 80)},
    "cust_mno_port_interval_min": {"fraud": ("<=", 3), "normal": (">", 80)},
    "cust_mno_port_interval_stddev": {"fraud": ("<=", 17), "normal": (">", 46.7654)},
    "cust_mno_port_inverse_mean_interarrival_velocity": {"fraud": ("<=", 0.0049), "normal": (">", 0.0113)},
    "cust_mno_port_k_event_velocity_3": {"fraud": (">=", 0.5), "normal": ("<", 0.037)},
    "cust_mno_port_monthly_cnt_std": {"fraud": ("<=", 0), "normal": (">", 0.1)},
    "cust_phone_number_change_cumulative_event_slope": {"fraud": (">=", 0.5), "normal": ("<", 0.3333)},
    "cust_phone_number_change_days_for_last_3_events": {"fraud": ("<=", 1), "normal": (">", 2)},
    "cust_phone_number_change_interval_cv": {"fraud": (">=", 1.4142), "normal": ("<", 1.1705)},
    "cust_phone_number_change_interval_max": {"fraud": ("<=", 0), "normal": (">", 3)},
    "cust_phone_number_change_interval_mean": {"fraud": ("<=", 0), "normal": (">", 3)},
    "cust_phone_number_change_interval_median": {"fraud": ("<=", 0), "normal": (">", 3)},
    "cust_phone_number_change_interval_min": {"fraud": ("<=", 0), "normal": (">", 3)},
    "cust_phone_number_change_interval_stddev": {"fraud": ("<=", 0), "normal": (">", 1.4142)},
    "cust_phone_number_change_inverse_mean_interarrival_velocity": {"fraud": (">=", 1), "normal": ("<", 0.5)},
    "cust_phone_number_change_k_event_velocity_3": {"fraud": (">=", 1.5), "normal": ("<", 0.0092)},
    "cust_account_suspension_cnt_1d_over_3d": {"fraud": (">=", 1), "normal": ("<=", 0)},
    "cust_phone_number_change_cnt_1d_over_3d": {"fraud": (">=", 1), "normal": ("<=", 0)},
    "cust_mno_port_cnt_1d_over_3d": {"fraud": (">=", 1), "normal": ("<=", 0)},
    "cust_account_suspension_days_for_last_3_events": {"fraud": ("<=", 56), "normal": (">", 80.5)},
    "cust_account_suspension_interval_cv": {"fraud": (">=", 1.1198), "normal": ("<", 0.7245)},
    "cust_account_suspension_interval_stddev": {"fraud": ("<=", 35.95), "normal": (">", 76.9865)},
    "cust_account_suspension_k_event_velocity_3": {"fraud": ("<=", 0.0144), "normal": (">", 0.0484)},
    "cust_account_suspension_cumulative_event_slope": {"fraud": ("<=", 0.0078), "normal": (">", 0.0134)},
    "cust_account_suspension_interval_max": {"fraud": (">=", 179), "normal": ("<", 96)},
    "cust_account_suspension_interval_mean": {"fraud": ("<=", 23), "normal": (">", 52.3)},
    "cust_account_suspension_interval_median": {"fraud": ("<=", 15), "normal": (">", 33.5)},
    "cust_account_suspension_interval_min": {"fraud": ("<=", 9), "normal": (">", 30.5)},
    "cust_account_suspension_inverse_mean_interarrival_velocity": {"fraud": ("<=", 0.0084), "normal": (">", 0.01455)},
    "cust_account_suspension_monthly_cnt_std": {"fraud": ("<=", 0), "normal": (">", 0.1)},
    "cust_sim_change_days_for_last_5_events": {"fraud": ("<=", 63), "normal": (">", 127.5)},
    "cust_sim_change_k_event_velocity_5": {"fraud": ("<=", 0.0181), "normal": (">", 0.0253)},
    "cust_sim_change_cnt_1d_over_3d": {"fraud": (">=", 1), "normal": ("<", 0.8737)},
    "cust_account_suspension_cnt_7d_over_30d": {"fraud": (">=", 1), "normal": ("<", 0.2762)},
    "cust_phone_number_change_cnt_7d_over_30d": {"fraud": (">=", 1), "normal": ("<", 0.3447)},
    "cust_mno_port_cnt_7d_over_30d": {"fraud": (">=", 1), "normal": ("<", 0.5403)},
    "cust_account_suspension_cnt_30d_over_90d": {"fraud": (">=", 1), "normal": ("<", 0.4521)},
    "cust_device_change_cnt_1d_over_3d": {"fraud": (">=", 1), "normal": ("<", 0.7204)},
    "cust_account_suspension_active_months": {"fraud": (">=", 3), "normal": ("<", 2)},
    "cust_account_suspension_any_1d": {"fraud": ("==", 1), "normal": ("==", 0)},
    "cust_account_suspension_any_30d": {"fraud": ("==", 1), "normal": ("==", 0)},
    "cust_account_suspension_any_3d": {"fraud": ("==", 1), "normal": ("==", 0)},
    "cust_account_suspension_any_7d": {"fraud": ("==", 1), "normal": ("==", 0)},
    "cust_account_suspension_any_90d": {"fraud": ("==", 1), "normal": ("==", 0)},
    "cust_account_suspension_avg_monthly_cnt": {"fraud": (">=", 1.25), "normal": ("<", 1.08)},
    "cust_account_suspension_cnt_1d": {"fraud": (">=", 1), "normal": ("<", 0.018)},
    "cust_account_suspension_cnt_30d": {"fraud": (">=", 1), "normal": ("<", 0.4598)},
    "cust_account_suspension_cnt_3d": {"fraud": (">=", 1), "normal": ("<", 0.062)},
    "cust_account_suspension_cnt_7d": {"fraud": (">=", 1), "normal": ("<", 0.1338)},
    "cust_account_suspension_cnt_90d": {"fraud": (">=", 2), "normal": ("<", 1)},
    "cust_account_suspension_days_since_last": {"fraud": ("<=", 12.5), "normal": (">", 41)},
    "cust_account_suspension_event_day_of_week_entropy": {"fraud": (">=", 1.5), "normal": ("<", 0.7336)},
    "cust_account_suspension_event_hour_entropy": {"fraud": (">=", 1), "normal": ("<", 0.5833)},
    "cust_account_suspension_event_month_entropy": {"fraud": (">=", 1.5), "normal": ("<", 0.7726)},
    "cust_account_suspension_last_ts": {"fraud": ("<=", 29), "normal": (">", 61)},
    "cust_account_suspension_max_monthly_cnt": {"fraud": (">=", 2), "normal": ("<", 1.142)},
    "cust_account_suspension_monthly_peak_to_avg": {"fraud": (">=", 1.75), "normal": ("<", 1.0435)},
    "cust_account_suspension_recency_weighted_intensity": {"fraud": (">=", 0.176), "normal": ("<", 0.1312)},
    "cust_enstream_api_call_cnt_1d_over_3d": {"fraud": (">=", 1), "normal": ("<", 0.1667)},
    "cust_sim_change_k_event_velocity_3": {"fraud": ("<=", 0.015), "normal": (">", 0.0203)},
    "cust_sim_change_interval_cv": {"fraud": (">=", 1.4142), "normal": ("<", 1.2598)},
    "cust_sim_change_days_for_last_3_events": {"fraud": ("<=", 16), "normal": (">", 40)},
    "cust_sim_change_interval_stddev": {"fraud": ("<=", 4.2426), "normal": (">", 9.8995)},
    "cust_phone_number_change_cnt_30d_over_90d": {"fraud": (">=", 1), "normal": ("<", 0.5491)},
    "cust_sim_change_cnt_7d_over_30d": {"fraud": (">=", 1), "normal": ("<", 0.5)},
    "cust_mno_port_cnt_30d_over_90d": {"fraud": (">=", 1), "normal": ("<", 0.6822)},
    "cust_sim_change_monthly_cnt_std": {"fraud": (">=", 0.7071), "normal": ("<", 0.4082)},
    "cust_sim_change_cumulative_event_slope": {"fraud": ("<=", 0.0049), "normal": (">", 0.0069)},
    "cust_sim_change_inverse_mean_interarrival_velocity": {"fraud": ("<=", 0.0051), "normal": (">", 0.0071)},
    "cust_phone_number_change_any_1d": {"fraud": ("==", 1), "normal": ("==", 0)},
    "cust_phone_number_change_any_30d": {"fraud": ("==", 1), "normal": ("==", 0)},
    "cust_phone_number_change_any_3d": {"fraud": ("==", 1), "normal": ("==", 0)},
    "cust_phone_number_change_any_7d": {"fraud": ("==", 1), "normal": ("==", 0)},
    "cust_phone_number_change_any_90d": {"fraud": ("==", 1), "normal": ("==", 0)},
    "cust_phone_number_change_avg_monthly_cnt": {"fraud": (">", 1), "normal": ("<=", 1)},
    "cust_phone_number_change_cnt_1d": {"fraud": (">=", 1), "normal": ("<", 0.0157)},
    "cust_phone_number_change_cnt_30d": {"fraud": (">=", 1), "normal": ("<", 0.2241)},
    "cust_phone_number_change_cnt_3d": {"fraud": (">=", 1), "normal": ("<", 0.0292)},
    "cust_phone_number_change_cnt_7d": {"fraud": (">=", 1), "normal": ("<", 0.0816)},
    "cust_phone_number_change_cnt_90d": {"fraud": (">=", 1), "normal": ("<", 0.3985)},
    "cust_phone_number_change_days_since_last": {"fraud": ("<=", 41), "normal": (">", 89)},
    "cust_phone_number_change_event_day_of_week_entropy": {"fraud": (">", 0), "normal": ("==", 0)},
    "cust_phone_number_change_event_hour_entropy": {"fraud": (">", 0), "normal": ("==", 0)},
    "cust_phone_number_change_last_ts": {"fraud": ("<=", 26), "normal": (">", 57)},
    "cust_phone_number_change_max_monthly_cnt": {"fraud": (">", 1), "normal": ("<=", 1)},
    "cust_phone_number_change_recency_weighted_intensity": {"fraud": (">=", 0.1018), "normal": ("<", 0.0173)},
    "cust_sim_change_interval_max": {"fraud": ("<=", 7), "normal": (">", 10.5)},
    "cust_sim_change_interval_mean": {"fraud": ("<=", 7), "normal": (">", 10)},
    "cust_sim_change_interval_median": {"fraud": ("<=", 7), "normal": (">", 8)},
    "cust_sim_change_interval_min": {"fraud": ("<=", 2), "normal": (">", 4.5)},
    "cust_enstream_api_call_cnt_7d_over_30d": {"fraud": (">=", 1), "normal": ("<", 0.5)},
    "cust_sim_change_cnt_30d_over_90d": {"fraud": (">=", 1), "normal": ("<", 0.5)},
    "cust_device_change_k_event_velocity_5": {"fraud": (">=", 0.2336), "normal": ("<", 0.1163)},
    "cust_device_change_days_for_last_5_events": {"fraud": ("<=", 14), "normal": (">", 65)},
    "cust_mno_port_active_months": {"fraud": (">", 1), "normal": ("<=", 1)},
    "cust_mno_port_any_1d": {"fraud": ("==", 1), "normal": ("==", 0)},
    "cust_mno_port_any_30d": {"fraud": ("==", 1), "normal": ("==", 0)},
    "cust_mno_port_any_3d": {"fraud": ("==", 1), "normal": ("==", 0)},
    "cust_mno_port_any_7d": {"fraud": ("==", 1), "normal": ("==", 0)},
    "cust_mno_port_any_90d": {"fraud": ("==", 1), "normal": ("==", 0)},
    "cust_mno_port_avg_monthly_cnt": {"fraud": (">", 1), "normal": ("<=", 1)},
    "cust_mno_port_cnt_1d": {"fraud": (">=", 1), "normal": ("<", 0.0264)},
    "cust_mno_port_cnt_30d": {"fraud": (">=", 1), "normal": ("<", 0.2811)},
    "cust_mno_port_cnt_3d": {"fraud": (">=", 1), "normal": ("<", 0.0389)},
    "cust_mno_port_cnt_7d": {"fraud": (">=", 1), "normal": ("<", 0.1529)},
    "cust_mno_port_cnt_90d": {"fraud": (">=", 1), "normal": ("<", 0.4104)},
    "cust_mno_port_days_since_last": {"fraud": ("<=", 22), "normal": (">", 30)},
    "cust_mno_port_event_day_of_week_entropy": {"fraud": (">", 0), "normal": ("==", 0)},
    "cust_mno_port_event_hour_entropy": {"fraud": (">=", 1), "normal": ("<", 0.0114)},
    "cust_mno_port_event_month_entropy": {"fraud": (">", 0), "normal": ("==", 0)},
    "cust_mno_port_last_ts": {"fraud": ("<=", 38), "normal": (">", 53)},
    "cust_mno_port_max_monthly_cnt": {"fraud": (">", 1), "normal": ("<=", 1)},
    "cust_mno_port_monthly_peak_to_avg": {"fraud": (">", 1), "normal": ("<=", 1)},
    "cust_mno_port_recency_weighted_intensity": {"fraud": (">=", 0.1747), "normal": ("<", 0.1132)},
    "cust_device_change_cnt_7d_over_30d": {"fraud": (">=", 1), "normal": ("<", 0.6165)},
}

OPS = {
    ">=": lambda c, v: c >= v,
    "<=": lambda c, v: c <= v,
    ">":  lambda c, v: c >  v,
    "<":  lambda c, v: c <  v,
    "==": lambda c, v: c == v,
}

def apply_rules(df, rules=RULES):
    """For each feature in rules, add <feature>_signal in {fraud, normal, uncertain, NULL}."""
    for feat, rule in rules.items():
        if feat not in df.columns:
            continue
        col = F.col(feat)
        f_op, f_val = rule["fraud"]
        n_op, n_val = rule["normal"]
        fraud_cond  = OPS[f_op](col, f_val)
        normal_cond = OPS[n_op](col, n_val)
        df = df.withColumn(
            f"{feat}_signal",
            F.when(fraud_cond,  "fraud")
             .when(normal_cond, "normal")
             .when(col.isNotNull(), "uncertain")
             .otherwise(None)
        )
    return df


Connection: project.spark.compatibility | Run start time: 2026-06-22 15:29:04.997005 | Run duration : 0:00:01.781833s.


In [3]:
%%pyspark project.spark.compatibility
# ─── Feature groups by fraud type × confidence ─────────────────────
# Each list = features tagged H/M/L for that fraud type in the spreadsheet

FPF_H = [
    "cust_account_suspension_k_event_velocity_3",
    "cust_account_suspension_cumulative_event_slope",
    "cust_account_suspension_cnt_7d_over_30d",
    "cust_account_suspension_cnt_30d_over_90d",
    "cust_account_suspension_active_months",
    "cust_account_suspension_any_1d",
    "cust_account_suspension_any_30d",
    "cust_account_suspension_any_3d",
    "cust_account_suspension_any_7d",
    "cust_account_suspension_any_90d",
    "cust_account_suspension_cnt_30d",
    "cust_account_suspension_cnt_3d",
    "cust_account_suspension_cnt_7d",
    "cust_account_suspension_cnt_90d",
    "cust_account_suspension_days_since_last",
    "cust_account_suspension_last_ts",
    "cust_account_suspension_max_monthly_cnt",
    "cust_account_suspension_monthly_peak_to_avg",
    "cust_account_suspension_recency_weighted_intensity",
    "cust_enstream_api_call_cnt_7d_over_30d",
    "cust_sim_change_cnt_30d_over_90d",
]
FPF_M = [
    "cust_account_suspension_days_for_last_5_events",
    "cust_account_suspension_k_event_velocity_5",
    "cust_account_suspension_cnt_1d_over_3d",
    "cust_account_suspension_days_for_last_3_events",
    "cust_account_suspension_interval_mean",
    "cust_account_suspension_interval_median",
    "cust_account_suspension_interval_min",
    "cust_account_suspension_inverse_mean_interarrival_velocity",
    "cust_account_suspension_avg_monthly_cnt",
]
FPF_L = [
    "cust_enstream_api_call_cnt_1d_over_3d",
]

TrueName_H = [
    "cust_mno_port_cumulative_event_slope",
    "cust_device_change_cnt_1d_over_3d",
    "cust_enstream_api_call_cnt_1d_over_3d",
    "cust_enstream_api_call_cnt_7d_over_30d",
]
TrueName_M = [
    "cust_mno_port_interval_stddev",
    "cust_mno_port_cnt_30d_over_90d",
    "cust_mno_port_last_ts",
    "cust_mno_port_recency_weighted_intensity",
]
TrueName_L = [
    "cust_phone_number_change_interval_mean",
]

Synthetic_H = [
    "cust_sim_change_monthly_cnt_std",
    "cust_enstream_api_call_cnt_7d_over_30d",
]
Synthetic_M = [
    "cust_mno_port_interval_cv",
    "cust_mno_port_interval_mean",
    "cust_account_suspension_event_month_entropy",
    "cust_mno_port_cnt_30d_over_90d",
    "cust_mno_port_last_ts",
    "cust_mno_port_recency_weighted_intensity",
]
Synthetic_L = [
    "cust_phone_number_change_interval_mean",
]

Mule_H = [
    "cust_account_suspension_interval_cv",
    "cust_account_suspension_interval_stddev",
    "cust_sim_change_days_for_last_5_events",
    "cust_sim_change_k_event_velocity_5",
    "cust_sim_change_cnt_1d_over_3d",
    "cust_device_change_cnt_1d_over_3d",
    "cust_account_suspension_active_months",
    "cust_enstream_api_call_cnt_1d_over_3d",
    "cust_sim_change_interval_stddev",
    "cust_sim_change_monthly_cnt_std",
    "cust_sim_change_cumulative_event_slope",
    "cust_sim_change_inverse_mean_interarrival_velocity",
    "cust_phone_number_change_recency_weighted_intensity",
    "cust_sim_change_interval_mean",
    "cust_sim_change_interval_median",
    "cust_sim_change_interval_min",
    "cust_enstream_api_call_cnt_7d_over_30d",
    "cust_sim_change_cnt_30d_over_90d",
    "cust_device_change_k_event_velocity_5",
    "cust_device_change_days_for_last_5_events",
    "cust_device_change_cnt_7d_over_30d",
]
Mule_M = [
    "cust_mno_port_interval_stddev",
    "cust_phone_number_change_interval_stddev",
    "cust_account_suspension_days_since_last",
    "cust_sim_change_k_event_velocity_3",
    "cust_sim_change_interval_cv",
    "cust_sim_change_days_for_last_3_events",
    "cust_phone_number_change_cnt_30d_over_90d",
    "cust_mno_port_cnt_30d_over_90d",
    "cust_phone_number_change_active_months",
    "cust_phone_number_change_any_1d",
    "cust_phone_number_change_any_30d",
    "cust_phone_number_change_any_3d",
    "cust_phone_number_change_any_7d",
    "cust_phone_number_change_any_90d",
    "cust_phone_number_change_avg_monthly_cnt",
    "cust_phone_number_change_cnt_1d",
    "cust_phone_number_change_cnt_30d",
    "cust_phone_number_change_cnt_3d",
    "cust_phone_number_change_cnt_7d",
    "cust_phone_number_change_cnt_90d",
    "cust_phone_number_change_days_since_last",
    "cust_phone_number_change_event_month_entropy",
    "cust_phone_number_change_last_ts",
    "cust_mno_port_active_months",
    "cust_mno_port_any_1d",
    "cust_mno_port_any_30d",
    "cust_mno_port_any_3d",
    "cust_mno_port_any_7d",
    "cust_mno_port_any_90d",
    "cust_mno_port_cnt_1d",
    "cust_mno_port_cnt_30d",
    "cust_mno_port_cnt_3d",
    "cust_mno_port_cnt_7d",
    "cust_mno_port_cnt_90d",
    "cust_mno_port_days_since_last",
    "cust_mno_port_event_month_entropy",
    "cust_mno_port_last_ts",
    "cust_mno_port_recency_weighted_intensity",
]
Mule_L = [
    "cust_phone_number_change_interval_mean",
]

ATO_H = [
    "cust_mno_port_cumulative_event_slope",
    "cust_mno_port_inverse_mean_interarrival_velocity",
    "cust_mno_port_k_event_velocity_3",
    "cust_mno_port_k_event_velocity_5",
    "cust_phone_number_change_cumulative_event_slope",
    "cust_phone_number_change_inverse_mean_interarrival_velocity",
    "cust_phone_number_change_k_event_velocity_3",
    "cust_phone_number_change_k_event_velocity_5",
    "cust_phone_number_change_cnt_1d_over_3d",
    "cust_mno_port_cnt_1d_over_3d",
    "cust_sim_change_days_for_last_5_events",
    "cust_sim_change_k_event_velocity_5",
    "cust_sim_change_cnt_1d_over_3d",
    "cust_device_change_cnt_1d_over_3d",
    "cust_enstream_api_call_cnt_1d_over_3d",
    "cust_sim_change_k_event_velocity_3",
    "cust_sim_change_interval_cv",
    "cust_sim_change_days_for_last_3_events",
    "cust_sim_change_interval_stddev",
    "cust_sim_change_cnt_7d_over_30d",
    "cust_sim_change_monthly_cnt_std",
    "cust_sim_change_cumulative_event_slope",
    "cust_sim_change_inverse_mean_interarrival_velocity",
    "cust_phone_number_change_any_1d",
    "cust_phone_number_change_any_30d",
    "cust_phone_number_change_any_3d",
    "cust_phone_number_change_any_7d",
    "cust_phone_number_change_any_90d",
    "cust_phone_number_change_cnt_1d",
    "cust_phone_number_change_cnt_30d",
    "cust_phone_number_change_cnt_3d",
    "cust_phone_number_change_cnt_7d",
    "cust_phone_number_change_cnt_90d",
    "cust_phone_number_change_days_since_last",
    "cust_phone_number_change_recency_weighted_intensity",
    "cust_sim_change_interval_mean",
    "cust_sim_change_interval_median",
    "cust_sim_change_interval_min",
    "cust_enstream_api_call_cnt_7d_over_30d",
    "cust_sim_change_cnt_30d_over_90d",
    "cust_mno_port_any_1d",
    "cust_mno_port_any_30d",
    "cust_mno_port_any_3d",
    "cust_mno_port_any_7d",
    "cust_mno_port_any_90d",
    "cust_mno_port_cnt_1d",
    "cust_mno_port_cnt_30d",
    "cust_mno_port_cnt_3d",
    "cust_mno_port_cnt_7d",
    "cust_mno_port_cnt_90d",
    "cust_mno_port_days_since_last",
    "cust_mno_port_recency_weighted_intensity",
]
ATO_M = [
    "cust_mno_port_interval_cv",
    "cust_mno_port_interval_mean",
    "cust_phone_number_change_interval_cv",
    "cust_phone_number_change_cnt_30d_over_90d",
    "cust_mno_port_cnt_30d_over_90d",
    "cust_device_change_cnt_7d_over_30d",
]
ATO_L = []  # (empty)


Connection: project.spark.compatibility | Run start time: 2026-06-22 15:29:06.784611 | Run duration : 0:00:00.290537s.


In [4]:
%%pyspark project.spark.compatibility
# Combined H/M/L: dedup + highest tier wins  (Combined = "all")
all_H = set(FPF_H + TrueName_H + Synthetic_H + Mule_H + ATO_H)
all_M = set(FPF_M + TrueName_M + Synthetic_M + Mule_M + ATO_M)
all_L = set(FPF_L + TrueName_L + Synthetic_L + Mule_L + ATO_L)
Combined_H = sorted(all_H)
Combined_M = sorted(all_M - all_H)
Combined_L = sorted(all_L - all_H - all_M)
print(f"Combined: H={len(Combined_H)} M={len(Combined_M)} L={len(Combined_L)}")

GROUPS = {
    "FPF":       {"H": FPF_H,       "M": FPF_M,       "L": FPF_L},
    "TrueName":  {"H": TrueName_H,  "M": TrueName_M,  "L": TrueName_L},
    "Synthetic": {"H": Synthetic_H, "M": Synthetic_M, "L": Synthetic_L},
    "Mule":      {"H": Mule_H,      "M": Mule_M,      "L": Mule_L},
    "ATO":       {"H": ATO_H,       "M": ATO_M,       "L": ATO_L},
    "Combined":  {"H": Combined_H,  "M": Combined_M,  "L": Combined_L},   # Combined = all
}
WEIGHTS_FRAUD = {"H": 3, "M": 2, "L": 1}

def add_type_scores_both(df, type_name, levels_dict):
    hits_fo, score_fo = F.lit(0), F.lit(0.0)
    hits_wu, score_wu = F.lit(0), F.lit(0.0)
    for level, feats in levels_dict.items():
        w_f = WEIGHTS_FRAUD[level]
        w_u = w_f * 0.5
        for feat in feats:
            sig = f"{feat}_signal"
            if sig not in df.columns:
                continue
            is_f = F.when(F.col(sig) == "fraud",     1).otherwise(0)
            is_u = F.when(F.col(sig) == "uncertain", 1).otherwise(0)
            hits_fo  = hits_fo  + is_f
            score_fo = score_fo + is_f * w_f
            hits_wu  = hits_wu  + is_f + is_u
            score_wu = score_wu + is_f * w_f + is_u * w_u
    return (df
        .withColumn(f"{type_name}_hits",                 hits_fo)
        .withColumn(f"{type_name}_score",                score_fo)
        .withColumn(f"{type_name}_hits_with_uncertain",  hits_wu)
        .withColumn(f"{type_name}_score_with_uncertain", score_wu))

score_cols = (
    [f"{t}_hits"                 for t in GROUPS.keys()] +
    [f"{t}_score"                for t in GROUPS.keys()] +
    [f"{t}_hits_with_uncertain"  for t in GROUPS.keys()] +
    [f"{t}_score_with_uncertain" for t in GROUPS.keys()]
)

def add_rule_scores(df):
    """Any df with the cust_* features -> adds <feat>_signal + the per-fraud-type hit/score columns.
       Fraud types: FPF, TrueName, Synthetic, Mule, ATO, Combined (= all)."""
    df = apply_rules(df)
    for type_name, levels in GROUPS.items():
        df = add_type_scores_both(df, type_name, levels)
    return df

Combined: H=76 M=24 L=1
Connection: project.spark.compatibility | Run start time: 2026-06-22 15:29:07.079285 | Run duration : 0:00:00.754813s.


In [5]:
%%pyspark project.spark.compatibility
# point at ANY dataset that has the cust_* behavioural features
INPUT_PATH = "s3://enstream-lake-gold-dev/poc/analysis/rule_based/v2_validation"

df        = spark.read.parquet(INPUT_PATH)
df_scored = add_rule_scores(df)          # auto-adds the 24 per-type columns

# view only — nothing is written
print(score_cols)
df_scored.select(*score_cols).describe().show()

['FPF_hits', 'TrueName_hits', 'Synthetic_hits', 'Mule_hits', 'ATO_hits', 'Combined_hits', 'FPF_score', 'TrueName_score', 'Synthetic_score', 'Mule_score', 'ATO_score', 'Combined_score', 'FPF_hits_with_uncertain', 'TrueName_hits_with_uncertain', 'Synthetic_hits_with_uncertain', 'Mule_hits_with_uncertain', 'ATO_hits_with_uncertain', 'Combined_hits_with_uncertain', 'FPF_score_with_uncertain', 'TrueName_score_with_uncertain', 'Synthetic_score_with_uncertain', 'Mule_score_with_uncertain', 'ATO_score_with_uncertain', 'Combined_score_with_uncertain']
+-------+-------------------+--------------------+--------------------+-------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+------------------+-----------------+-----------------------+----------------------------+-----------------------------+------------------------+-----------------------+----------------------------+------------------------+--------------------

In [8]:
%%pyspark project.spark.compatibility
df_scored.show(1, False, True)


-RECORD 0------------------------------------------------------------------------------------------------------------------------------
 cust_phone_number_change_to_status_change_s_avg_transition_days    | NULL                                                             
 cust_status_change_a_to_phone_number_change_avg_transition_days    | NULL                                                             
 cust_sim_change_to_device_change_transition_velocity               | NULL                                                             
 cust_device_change_to_enstream_api_call_transition_velocity        | NULL                                                             
 cust_status_change_c_to_status_change_a_transition_velocity        | NULL                                                             
 cust_status_change_s_to_status_change_c_transition_velocity        | NULL                                                             
 cust_status_change_s_to_enstream_api_call_min_t

In [ ]:
%%pyspark project.spark.compatibility

